In [6]:
conda install -c conda-forge lightgbm -y

Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3

  added / updated specs:
    - lightgbm


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    boost-cpp-1.84.0           |       hca5e981_3          16 KB  conda-forge
    ca-certificates-2026.5.20  |       hbd8a1cb_0         127 KB  conda-forge
    certifi-2026.5.20          |     pyhd8ed1ab_0         131 KB  conda-forge
    conda-24.11.3              |  py312h81bd7bf_0         1.1 MB  conda-forge
    icu-73.2                   |       hc8870d7_0        11.4 MB  conda-forge
    khronos-opencl-icd-loader-2024.10.24|       h5505292_1          34 KB  conda-forge
    libboost-1.84.0            |       h17eb2be_3         1.9 MB  conda-forge
    libboost-devel-1.84.0      |       hf450f58_3          39 KB  conda-forge
    libboost-headers-1.84.0    |     

In [9]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
from scipy.optimize import minimize_scalar
print("start")

# ─────────────────────────────────────────
# 1. LOAD & BASE FEATURE ENGINEERING
# ─────────────────────────────────────────
df = pd.read_csv('dataset/train.csv')
df = df.drop(columns=['Index'], errors='ignore')

def engineer_features(df):
    df = df.copy()
    df[['Hour', 'Minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
    df['Hour_sin']        = np.sin(2 * np.pi * df['Hour']   / 24)
    df['Hour_cos']        = np.cos(2 * np.pi * df['Hour']   / 24)
    df['Minute_sin']      = np.sin(2 * np.pi * df['Minute'] / 60)
    df['Minute_cos']      = np.cos(2 * np.pi * df['Minute'] / 60)
    df['day_of_week']     = df['day'] % 7
    df['is_weekend']      = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_morning_rush'] = df['Hour'].isin([8, 9, 10]).astype(int)
    df['is_evening_rush'] = df['Hour'].isin([17, 18, 19, 20]).astype(int)
    df = df.drop(columns=['timestamp'])
    return df

df = engineer_features(df)
df = df.sort_values(['geohash', 'day', 'Hour', 'Minute']).reset_index(drop=True)

# ─────────────────────────────────────────
# 2. MISSINGNESS FLAGS
# ─────────────────────────────────────────
df['is_Temp_missing']     = df['Temperature'].isna().astype(int)
df['is_Weather_missing']  = df['Weather'].isna().astype(int)
df['is_RoadType_missing'] = df['RoadType'].isna().astype(int)

# ─────────────────────────────────────────
# 3. LAG FEATURES
# ─────────────────────────────────────────
def add_lag_features(df):
    g = df.groupby('geohash')['demand']
    for lag in [1, 2, 3, 4, 8, 12, 48, 96, 672]:
        df[f'demand_lag_{lag}'] = g.shift(lag)
    for win in [3, 6, 12, 24]:
        df[f'rolling_mean_{win}'] = g.transform(
            lambda x: x.shift(1).rolling(win, min_periods=1).mean())
        df[f'rolling_std_{win}'] = g.transform(
            lambda x: x.shift(1).rolling(win, min_periods=1).std().fillna(0))
    return df

df = add_lag_features(df)

# ─────────────────────────────────────────
# 4. GEO-HOUR PROFILE (shift-based, no leakage)
# ─────────────────────────────────────────
df['geo_hour_hist_mean'] = df.groupby(['geohash', 'Hour'])['demand'].transform(
    lambda x: x.shift(96).expanding().mean()
)
global_hour_mean = df.groupby('Hour')['demand'].mean()

def fill_geo_hour(row):
    if pd.isna(row['geo_hour_hist_mean']):
        return global_hour_mean[row['Hour']]
    return row['geo_hour_hist_mean']

df['geo_hour_hist_mean'] = df.apply(fill_geo_hour, axis=1)

# ─────────────────────────────────────────
# 5. GEOHASH DECODE → LAT/LON
# ─────────────────────────────────────────
def decode_geohash(gh):
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_range, lon_range = [-90.0, 90.0], [-180.0, 180.0]
    is_lon = True
    for char in gh:
        bits = base32.index(char)
        for shift in range(4, -1, -1):
            bit = (bits >> shift) & 1
            if is_lon:
                mid = sum(lon_range) / 2
                lon_range[0 if bit else 1] = mid
            else:
                mid = sum(lat_range) / 2
                lat_range[0 if bit else 1] = mid
            is_lon = not is_lon
    return (sum(lat_range) / 2, sum(lon_range) / 2)

unique_geohashes = df['geohash'].drop_duplicates()
latlon_decoded   = unique_geohashes.apply(decode_geohash)
latlon_df        = pd.DataFrame(latlon_decoded.tolist(), columns=['lat', 'lon'])
latlon_df['geohash'] = unique_geohashes.values
df = df.merge(latlon_df, on='geohash', how='left')

# ─────────────────────────────────────────
# 6. CATEGORICAL ENCODING
# ─────────────────────────────────────────
df['geohash_macro_4'] = df['geohash'].str[:4]
df['geohash_macro_5'] = df['geohash'].str[:5]

cat_cols = ['geohash', 'geohash_macro_4', 'geohash_macro_5',
            'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in cat_cols:
    df[col] = df[col].fillna('Unknown').astype(str)

# ─────────────────────────────────────────
# 7. DEFINE FEATURES
# ─────────────────────────────────────────
FEATURE_COLS = [c for c in df.columns if c not in ['demand', 'day', 'demand_lag_672']]
X, y = df[FEATURE_COLS], df['demand']

# ─────────────────────────────────────────
# 8. TIME-SERIES CROSS-VALIDATION
# ─────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5)

cb_scores, lgbm_scores = [], []
cb_models, lgbm_models = [], []

last_cb_oof, last_lgbm_oof, last_y_val = None, None, None

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr,  X_val = X.iloc[train_idx],  X.iloc[val_idx]
    y_tr,  y_val = y.iloc[train_idx],  y.iloc[val_idx]

    # ── CatBoost ──
    cb = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5,
        cat_features=cat_cols,
        eval_metric='R2',
        random_seed=42,
        verbose=200,
        early_stopping_rounds=50
    )
    cb.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    cb_preds = cb.predict(X_val)
    cb_r2    = max(0, r2_score(y_val, cb_preds)) * 100
    cb_scores.append(cb_r2)
    cb_models.append(cb)

    # ── LightGBM ──
    X_tr_lgbm  = X_tr.copy()
    X_val_lgbm = X_val.copy()
    for col in cat_cols:
        X_tr_lgbm[col]  = X_tr_lgbm[col].astype('category')
        X_val_lgbm[col] = X_val_lgbm[col].astype('category')

    lgbm = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=127,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    )
    lgbm.fit(
        X_tr_lgbm, y_tr,
        eval_set=[(X_val_lgbm, y_val)],
        eval_metric='r2',                      # FIXED: must be here, not in constructor
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(200)
        ]
    )
    lgbm_preds = lgbm.predict(X_val_lgbm)
    lgbm_r2    = max(0, r2_score(y_val, lgbm_preds)) * 100
    lgbm_scores.append(lgbm_r2)
    lgbm_models.append(lgbm)

    print(f"Fold {fold+1} | CB R²: {cb_r2:.2f} | LGBM R²: {lgbm_r2:.2f}")

    last_cb_oof   = cb_preds
    last_lgbm_oof = lgbm_preds
    last_y_val    = y_val

print(f"\nMean CV — CatBoost: {np.mean(cb_scores):.2f} | LGBM: {np.mean(lgbm_scores):.2f}")

# ─────────────────────────────────────────
# 9. OPTIMAL BLEND WEIGHT
# ─────────────────────────────────────────
def neg_r2_blend(w):
    blended = w * last_cb_oof + (1 - w) * last_lgbm_oof
    return -(max(0, r2_score(last_y_val, blended)) * 100)

result = minimize_scalar(neg_r2_blend, bounds=(0, 1), method='bounded')
best_w = result.x
print(f"\nOptimal blend — CatBoost: {best_w:.3f} | LGBM: {1 - best_w:.3f}")
print(f"Blended R² on last fold: {-result.fun:.2f}")

# ─────────────────────────────────────────
# 10. FINAL MODELS ON FULL TRAINING DATA
# ─────────────────────────────────────────
best_cb_iters   = int(np.mean([m.get_best_iteration() for m in cb_models]))
best_lgbm_iters = int(np.mean([m.best_iteration_ for m in lgbm_models]))
print(f"\nFinal iterations — CatBoost: {best_cb_iters} | LGBM: {best_lgbm_iters}")

final_cb = CatBoostRegressor(
    iterations=best_cb_iters,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    cat_features=cat_cols,
    random_seed=42,
    verbose=200
)
final_cb.fit(X, y)

X_lgbm = X.copy()
for col in cat_cols:
    X_lgbm[col] = X_lgbm[col].astype('category')

final_lgbm = LGBMRegressor(
    n_estimators=best_lgbm_iters,
    learning_rate=0.03,
    num_leaves=127,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=100
)
final_lgbm.fit(X_lgbm, y)          # no early_stopping here, fixed iterations

# ─────────────────────────────────────────
# 11. TEST LOADING & FEATURE ENGINEERING
# ─────────────────────────────────────────
test_raw       = pd.read_csv('dataset/test.csv')
test_index_col = test_raw['Index'].values

test_df = engineer_features(test_raw)
test_df['is_Temp_missing']     = test_df['Temperature'].isna().astype(int)
test_df['is_Weather_missing']  = test_df['Weather'].isna().astype(int)
test_df['is_RoadType_missing'] = test_df['RoadType'].isna().astype(int)
test_df['is_test'] = 1
test_df['demand'] = np.nan

test_df = test_df.merge(latlon_df, on='geohash', how='left')
test_df['geohash_macro_4'] = test_df['geohash'].str[:4]
test_df['geohash_macro_5'] = test_df['geohash'].str[:5]
for col in cat_cols:
    test_df[col] = test_df[col].fillna('Unknown').astype(str)

# ─────────────────────────────────────────
# 12. COMBINE TRAIN + TEST FOR ITERATIVE FORECASTING
# ─────────────────────────────────────────

LAG_COLS     = [1, 2, 3, 4, 8, 12, 48, 96]   # DROP 672 — too sparse
ROLL_WINDOWS = [3, 6, 12, 24]

# Use FULL training df as history seed (not just tail)
# Keep only columns needed to seed lags — demand + sort keys
train_history = df[['geohash', 'day', 'Hour', 'Minute', 'demand']].copy()
train_history['is_test'] = 0

test_df['demand']  = np.nan
test_df['is_test'] = 1

# Shared static feature cols (non-lag)
static_cols = [
    'geohash', 'day', 'Hour', 'Minute', 'demand', 'is_test',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather',
    'Hour_sin', 'Hour_cos', 'Minute_sin', 'Minute_cos',
    'day_of_week', 'is_weekend', 'is_morning_rush', 'is_evening_rush',
    'is_Temp_missing', 'is_Weather_missing', 'is_RoadType_missing',
    'lat', 'lon', 'geohash_macro_4', 'geohash_macro_5'
]

# For train_history rows, static cols are NaN (we only need demand for lag seeding)
# For test rows, static cols are fully populated
combined = pd.concat(
    [train_history,                                          # full train history for lag seeding
     test_df[[c for c in static_cols if c in test_df.columns]]],
    ignore_index=True
)
combined = combined.sort_values(['geohash', 'day', 'Hour', 'Minute']).reset_index(drop=True)

# Initialize lag/rolling cols as NaN — will be filled per-row in loop
lag_and_roll_cols = (
    [f'demand_lag_{lag}' for lag in LAG_COLS] +
    [f'rolling_mean_{win}' for win in ROLL_WINDOWS] +
    [f'rolling_std_{win}'  for win in ROLL_WINDOWS] +
    ['geo_hour_hist_mean']
)
for col in lag_and_roll_cols:
    if col not in combined.columns:
        combined[col] = np.nan

# Fix lat/lon for any test geohashes not in training
missing_latlon = combined['lat'].isna()
if missing_latlon.any():
    new_ghs = combined.loc[missing_latlon, 'geohash'].dropna().unique()
    print(f"Decoding {len(new_ghs)} unseen geohashes for lat/lon...")
    for gh in new_ghs:
        lat, lon = decode_geohash(gh)
        combined.loc[combined['geohash'] == gh, 'lat'] = lat
        combined.loc[combined['geohash'] == gh, 'lon'] = lon

# Fill missing Temperature with training median
temp_median = df['Temperature'].median()
combined['Temperature'] = combined['Temperature'].fillna(temp_median)

# Fill categorical NaNs
for col in cat_cols:
    combined[col] = combined[col].fillna('Unknown').astype(str)

print(f"Combined shape: {combined.shape}")
print(f"Train rows   : {(combined['is_test']==0).sum()}")
print(f"Test rows    : {(combined['is_test']==1).sum()}")

# ─────────────────────────────────────────
# 13. ITERATIVE FORECASTING LOOP
# ─────────────────────────────────────────
gh_index_map = {
    gh: grp.index.tolist()
    for gh, grp in combined.groupby('geohash')
}

test_indices = combined[combined['is_test'] == 1].index.tolist()

# FEATURE_COLS must exclude 'day', 'demand', 'is_test'
# and must NOT include demand_lag_672 anymore
FEATURE_COLS = [c for c in df.columns
                if c not in ['demand', 'day', 'demand_lag_672']]
# Add lag/roll/geo cols that weren't in df originally
for col in lag_and_roll_cols:
    if col not in FEATURE_COLS and col != 'demand_lag_672':
        FEATURE_COLS.append(col)
# Remove any duplicates
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))

print(f"\nFeature count: {len(FEATURE_COLS)}")
print(f"Iterative forecasting over {len(test_indices)} rows...")

for i, idx in enumerate(test_indices):
    if i % 5000 == 0:
        print(f"  {i}/{len(test_indices)}")

    gh      = combined.at[idx, 'geohash']
    gh_rows = gh_index_map[gh]
    pos     = gh_rows.index(idx)

    # Lag features
    for lag in LAG_COLS:
        lag_pos = pos - lag
        combined.at[idx, f'demand_lag_{lag}'] = (
            combined.at[gh_rows[lag_pos], 'demand']
            if lag_pos >= 0 else np.nan
        )

    # Rolling stats
    for win in ROLL_WINDOWS:
        past = [
            combined.at[gh_rows[pos - k], 'demand']
            for k in range(1, min(win + 1, pos + 1))
            if not pd.isna(combined.at[gh_rows[pos - k], 'demand'])
        ]
        combined.at[idx, f'rolling_mean_{win}'] = np.mean(past) if past else 0.0
        combined.at[idx, f'rolling_std_{win}']  = np.std(past)  if len(past) > 1 else 0.0

    # Geo-hour mean
    combined.at[idx, 'geo_hour_hist_mean'] = global_hour_mean.get(
        combined.at[idx, 'Hour'], df['demand'].mean()
    )

    # Predict
    row = combined.loc[[idx], FEATURE_COLS].copy()
    for col in cat_cols:
        row[col] = row[col].fillna('Unknown').astype(str)

    row_lgbm = row.copy()
    for col in cat_cols:
        row_lgbm[col] = row_lgbm[col].astype('category')

    pred_cb   = final_cb.predict(row)[0]
    pred_lgbm = final_lgbm.predict(row_lgbm)[0]
    pred      = best_w * pred_cb + (1 - best_w) * pred_lgbm

    combined.at[idx, 'demand'] = max(0.0, pred)

# ─────────────────────────────────────────
# SANITY CHECK
# ─────────────────────────────────────────
test_rows    = combined[combined['is_test'] == 1][FEATURE_COLS]
nan_frac     = test_rows.isna().mean()
problem_cols = nan_frac[nan_frac > 0].sort_values(ascending=False)
if len(problem_cols) > 0:
    print("\n⚠ WARNING — NaN features remaining:")
    print(problem_cols.head(15))
else:
    print("\n✓ No NaN features — good to go")

# ─────────────────────────────────────────
# 14. SAVE SUBMISSION
# ─────────────────────────────────────────
final_test = combined[combined['is_test'] == 1].copy()

# Align with original test order
submission = pd.DataFrame({
    'Index':  test_index_col,
    'demand': final_test.sort_values(['geohash', 'day', 'Hour', 'Minute'])['demand'].values
})

# Re-sort by Index to match expected submission order
submission['demand'] = submission['demand'].clip(lower=0)
submission = submission.sort_values('Index').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)

print(f"\nDone.")
print(f"Rows       : {len(submission)}")
print(f"Mean demand: {submission['demand'].mean():.4f}")
print(f"Min  demand: {submission['demand'].min():.4f}")
print(f"Max  demand: {submission['demand'].max():.4f}")

start
0:	learn: 0.0483508	test: -0.0034529	best: -0.0034529 (0)	total: 52ms	remaining: 1m 43s
200:	learn: 0.9364754	test: 0.9067314	best: 0.9067314 (200)	total: 2.25s	remaining: 20.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9084951563
bestIteration = 332

Shrink model to first 333 iterations.
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.933623
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.269673
[LightGBM] [Debug] init for col-wise cost 0.001158 seconds, init for row-wise cost 0.001692 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 4562
[LightGBM] [Info] Number of data points in the train set: 12884, number of used features: 41
[LightGBM] [Info] Star

In [10]:
print("hi")

hi
